In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install numpy==1.26.4
!pip install -q tree-sitter==0.21.3 tree-sitter-python==0.21.0 tree-sitter-java==0.21.0
!pip install -q node2vec==0.4.6
!pip install -q transformers==4.41.0 sentencepiece networkx gradio
!pip install faiss-cpu

print("✅ Kurulum tamamlandı!")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.6/130.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
momepy 0.11.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
mapclassify 2.10.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
spopt 0.7.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.4 

In [ ]:
import json

with open('/content/drive/MyDrive/CodeReviewBot/model_ciktisi/metrikler.json') as f:
    metrikler = json.load(f)

bleu      = metrikler.get('bleu', 0)
bertscore = metrikler.get('bertscore_f1', 0)
f1        = metrikler.get('f1', 0)

print(f"BLEU: {bleu}, BERTScore: {bertscore}, F1: {f1}")

BLEU: 1.27, BERTScore: 0.8514, F1: 0.07


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/CodeReviewBot/utils/')

exec(open('/content/drive/MyDrive/CodeReviewBot/utils/pipeline.py').read())

print("✅ Pipeline hazır!")

Pipeline efsanevi sekilde hazir!
✅ Pipeline hazır!


In [ ]:
import re

def code_repair_kural(code, lang='python'):
    duzeltmeler = []
    yeni_kod = code

    # 1. SQL Injection
    if 'execute(' in code and ('" +' in code or "' +" in code):
        yeni_kod = re.sub(
            r'\.execute\(["\'](.+?)["\'] \+ (\w+)\)',
            r'.execute("\1?", (\2,))',
            yeni_kod
        )
        duzeltmeler.append("SQL injection açığı düzeltildi — parameterized query kullanıldı")

    # 2. Dosya kapatılmıyor
    if 'open(' in code and 'close()' not in code and 'with open' not in code:
        yeni_kod = re.sub(
            r'(\w+) = open\((.+?)\)',
            r'\1 = open(\2)',
            yeni_kod
        )
        yeni_kod = yeni_kod.replace(
            'return icerik',
            'icerik = f.read()\n    f.close()\n    return icerik'
        )
        duzeltmeler.append("Dosya kapatma eklendi — f.close() ile kaynaklar serbest bırakıldı")

    # 3. Sıfıra bölme
    if '/ sayi' in code or '/sayi' in code:
        yeni_kod = yeni_kod.replace(
            'sonuc = 100 / sayi',
            'if sayi == 0:\n            continue\n        sonuc = 100 / sayi'
        )
        duzeltmeler.append("Sıfıra bölme koruması eklendi")

    # 4. Sabit şifre
    if any(k in code for k in ['"1234"', "'1234'", '"admin"', "'admin'"]):
        yeni_kod = "# ⚠️ Sabit şifre güvenlik açığı! Environment variable kullanın.\n" + yeni_kod
        duzeltmeler.append("Sabit şifre tespit edildi — environment variable kullanılmalı")

    if not duzeltmeler:
        return code, ["Bilinen hata kalıbı bulunamadı — model yorumuna bakın"]

    return yeni_kod, duzeltmeler

print("✅ code_repair_kural hazır!")

✅ code_repair_kural hazır!


In [ ]:
import gradio as gr

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');
* { font-family: 'Inter', sans-serif !important; box-sizing: border-box; }

/* ── DARK (varsayılan) ── */
body, .gradio-container {
    background: #0a0a14 !important;
    margin: 0 !important; padding: 0 !important;
    max-width: 100% !important; width: 100% !important;
}

.main.svelte-1kyws56 { padding: 0 !important; }
.contain.svelte-1kyws56 { max-width: 100% !important; }
footer { display: none !important; }

/* ── LIGHT MODE — body.light-mode olunca tetiklenir ── */
body.light-mode,
body.light-mode .gradio-container { background: #f8fafc !important; }

body.light-mode textarea {
    background: #ffffff !important;
    color: #0f172a !important;
    border-color: #e2e8f0 !important;
}

body.light-mode .chatbot-alan,
body.light-mode .chatbot-alan > div,
body.light-mode [class*="chatbot"],
body.light-mode [class*="bubble"],
body.light-mode [class*="message"] {
    background: #ffffff !important;
    color: #0f172a !important;
    border-color: #e2e8f0 !important;
}

body.light-mode .metric-card {
    background: #ffffff;
    border-color: #e2e8f0;
}

body.light-mode [class*="svelte"] {
    background: transparent !important;
}

/* ── BUTONLAR ── */
.analiz-btn button {
    background: linear-gradient(135deg, #7C3AED, #4F46E5) !important;
    color: white !important; border: none !important;
    border-radius: 10px !important; font-size: 15px !important;
    font-weight: 600 !important; padding: 14px !important;
    width: 100% !important; cursor: pointer !important;
}
.analiz-btn button:hover { opacity: 0.88 !important; }

.send-btn button {
    background: #7C3AED !important; color: white !important;
    border: none !important; border-radius: 10px !important;
    font-weight: 600 !important; padding: 10px 18px !important;
    cursor: pointer !important;
}

/* ── TEXTAREA (dark default) ── */
textarea {
    background: #0a0a14 !important; color: #e2e8f0 !important;
    border: 1px solid #1e1e2e !important; border-radius: 10px !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 13px !important; line-height: 1.7 !important;
}

/* ── METRİK ── */
.metric-card {
    background: #0f0f1a; border: 1px solid #1e1e2e;
    border-radius: 12px; padding: 14px 16px;
}
.issue-badge { padding: 2px 8px; border-radius: 4px; font-size: 11px; font-weight: 600; }
.badge-perf { background: #f9731622; color: #f97316; }
.badge-read { background: #3b82f622; color: #3b82f6; }
.badge-sec  { background: #ef444422; color: #ef4444; }
.metric-bar-bg { height: 5px; background: #1e1e2e; border-radius: 3px; overflow: hidden; margin-top: 4px; }
.metric-bar-fill { height: 100%; border-radius: 3px; }
"""

def metrik_html_olustur():
    f1_pct   = round(f1 * 100)
    bert_pct = round(bertscore * 100)
    bleu_pct = min(round(bleu * 100), 100)
    return f"""
    <div style="display:flex;align-items:center;gap:16px;">
        <div style="flex:1;min-width:0;">
            <div style="display:flex;justify-content:space-between;
                        color:#94a3b8;font-size:12px;margin-bottom:4px;">
                BERTScore <span style="color:#7C3AED;font-weight:600;">{bertscore:.2f}</span>
            </div>
            <div class="metric-bar-bg">
                <div class="metric-bar-fill"
                     style="width:{bert_pct}%;background:linear-gradient(90deg,#7C3AED,#4F46E5);">
                </div>
            </div>
            <div style="display:flex;justify-content:space-between;
                        color:#94a3b8;font-size:12px;margin:8px 0 4px 0;">
                BLEU <span style="color:#06b6d4;font-weight:600;">{bleu:.2f}</span>
            </div>
            <div class="metric-bar-bg">
                <div class="metric-bar-fill"
                     style="width:{bleu_pct}%;background:linear-gradient(90deg,#06b6d4,#3b82f6);">
                </div>
            </div>
            <div style="display:flex;justify-content:space-between;
                        color:#94a3b8;font-size:12px;margin:8px 0 4px 0;">
                F1 Score <span style="color:#22c55e;font-weight:600;">{f1:.2f}</span>
            </div>
            <div class="metric-bar-bg">
                <div class="metric-bar-fill"
                     style="width:{f1_pct}%;background:linear-gradient(90deg,#22c55e,#06b6d4);">
                </div>
            </div>
        </div>
    </div>
    """

def analiz_et(kod, gecmis):
    if not kod.strip():
        return gecmis, gecmis, metrik_html_olustur()
    try:
        sonuc = code_review(kod, 'python')
        cevap = f"""*🔍 Analysis*
{sonuc["review"]}

*📊 Kod İstatistikleri*
- Graf düğüm sayısı: {sonuc["graph_nodes"]}
- Graf kenar sayısı: {sonuc["graph_edges"]}
- Kategori: {sonuc["kategori"]}

*🔎 Benzer Geçmiş İncelemeler (RAG)*
{chr(10).join(['• ' + r[:80] for r in sonuc["rag_ornekler"]])}"""
    except Exception as e:
        cevap = f"⚠️ Hata oluştu: {str(e)}"

    gecmis = gecmis + [
        {"role": "user", "content": kod},
        {"role": "assistant", "content": cevap}
    ]
    return gecmis, gecmis, metrik_html_olustur()

def takip_sorusu(mesaj, gecmis):
    if not mesaj.strip():
        return gecmis, gecmis, ""
    try:
        mesaj_lower = mesaj.lower()
        if any(k in mesaj_lower for k in ['düzelt', 'fix', 'repair', 'onar', 'düzeltirim', 'correct']):
            son_kod = ""

            for msg in reversed(gecmis):
                if msg["role"] == "user" and len(msg["content"]) > 20:
                    son_kod = msg["content"]
                    break

            if son_kod:
                # Dil tespiti
                if "public class" in son_kod or "System.out" in son_kod or "void main" in son_kod:
                    dil = "java"
                else:
                    dil = "python"

                # Doğru fonksiyonu çağır
                if dil == "java":
                    duzeltilmis, aciklamalar = code_repair(son_kod, 'java')
                else:
                    duzeltilmis, aciklamalar = code_repair_kural(son_kod, 'python')

                cevap = f"""*🔧 Suggested Fix*

```{dil}
{duzeltilmis}
```

*✅ Yapılan değişiklikler:*
{chr(10).join(['• ' + a for a in aciklamalar])}"""
            else:
                cevap = "Düzeltilecek kod bulunamadı. Önce sol tarafa kodu yapıştırın ve Analyze Code'a basın."
        else:
            sonuc = code_review(mesaj, 'python')
            cevap = sonuc["review"]
    except Exception as e:
        cevap = f"⚠️ Hata: {str(e)}"

    gecmis = gecmis + [
        {"role": "user", "content": mesaj},
        {"role": "assistant", "content": cevap}
    ]
    return gecmis, gecmis, ""

def temizle(gecmis):
    return [], []

with gr.Blocks(css=CSS, title="AI Code Reviewer") as demo:
    state = gr.State([])

    gr.HTML("""
    <div id="header" style="background:#0f0f1a;border-bottom:1px solid #1e1e2e;
                padding:12px 24px;display:flex;align-items:center;
                justify-content:space-between;width:100%;box-sizing:border-box;
                position:sticky;top:0;z-index:100;">
        <div style="display:flex;align-items:center;gap:12px;">
            <div style="background:linear-gradient(135deg,#7C3AED,#4F46E5);width:32px;height:32px;
                        border-radius:8px;display:flex;align-items:center;justify-content:center;
                        font-size:16px;flex-shrink:0;">✦</div>
            <span id="header-title" style="color:#e2e8f0;font-size:18px;font-weight:700;">AI Code Reviewer</span>
            <span style="color:#475569;font-size:11px;padding:3px 8px;border:1px solid #1e1e2e;border-radius:4px;">GraphBERT</span>
            <span style="color:#475569;font-size:11px;padding:3px 8px;border:1px solid #1e1e2e;border-radius:4px;">CodeT5+</span>
            <span style="color:#475569;font-size:11px;padding:3px 8px;border:1px solid #1e1e2e;border-radius:4px;">RAG</span>
        </div>
        <button onclick="toggleTheme()" id="theme-btn"
            style="background:#1a1a2e;border:1px solid #2d2d3a;border-radius:8px;
                   padding:6px 14px;color:#94a3b8;font-size:13px;cursor:pointer;">
            🌙 Dark
        </button>
    </div>

    <script>
    var isLight = false;

    function toggleTheme() {
        isLight = !isLight;
        var btn    = document.getElementById('theme-btn');
        var header = document.getElementById('header');
        var title  = document.getElementById('header-title');

        /* tek bir class toggle — tüm CSS kuralları buradan tetiklenir */
        document.body.classList.toggle('light-mode', isLight);

        if (isLight) {
            /* Header */
            header.style.background       = '#ffffff';
            header.style.borderBottomColor = '#e2e8f0';
            if (title) title.style.color  = '#0f172a';

            /* Buton */
            btn.innerHTML          = '☀️ Light';
            btn.style.background   = '#f1f5f9';
            btn.style.color        = '#475569';
            btn.style.borderColor  = '#e2e8f0';
        } else {
            header.style.background       = '#0f0f1a';
            header.style.borderBottomColor = '#1e1e2e';
            if (title) title.style.color  = '#e2e8f0';

            btn.innerHTML          = '🌙 Dark';
            btn.style.background   = '#1a1a2e';
            btn.style.color        = '#94a3b8';
            btn.style.borderColor  = '#2d2d3a';
        }
    }
    </script>
    """)

    with gr.Row():
        with gr.Column(scale=1, elem_classes="icerik-col"):
            with gr.Row(equal_height=True):
                with gr.Column(scale=1, min_width=0):
                    gr.HTML("""
                    <div style="background:#0f0f1a;border:1px solid #1e1e2e;
                                border-radius:14px 14px 0 0;padding:12px 16px;
                                display:flex;align-items:center;justify-content:space-between;">
                        <span style="color:#94a3b8;font-size:13px;font-weight:600;">🧠 Code Input</span>

                    </div>
                    """)
                    kod_girisi = gr.Textbox(
                        label="", lines=20,
                        placeholder="# Kodunuzu buraya yapıştırın...",
                        show_label=False
                    )
                    with gr.Row(elem_classes="analiz-btn"):
                        analiz_btn = gr.Button("🚀 Analyze Code")

                with gr.Column(scale=1, min_width=0):
                    gr.HTML("""
                    <div style="background:#0f0f1a;border:1px solid #1e1e2e;
                                border-radius:14px 14px 0 0;padding:12px 16px;
                                display:flex;align-items:center;justify-content:space-between;
                                width:100%;">
                        <span style="color:#94a3b8;font-size:13px;font-weight:600;">💬 AI Code Review Assistant</span>
                        <span onclick="clearChat()"
                              style="color:#475569;font-size:12px;cursor:pointer;
                                     padding:4px 10px;border:1px solid #2d2d3a;border-radius:6px;"
                              onmouseover="this.style.color='#ef4444';this.style.borderColor='#ef4444'"
                              onmouseout="this.style.color='#475569';this.style.borderColor='#2d2d3a'">
                            🗑️ Clear Chat
                        </span>
                    </div>
                    """)
                    chatbot = gr.Chatbot(
                        label="", height=430,
                        bubble_full_width=False,
                        show_label=False, type="messages",
                        elem_classes="chatbot-alan"
                    )
                    with gr.Row():
                        mesaj_kutusu = gr.Textbox(
                            placeholder="'neden?', 'nasıl düzeltirim?' gibi sorular yazın",
                            label="", scale=5, container=False
                        )
                        with gr.Column(scale=1, min_width=80, elem_classes="send-btn"):
                            sor_btn = gr.Button("Gönder")
                    temizle_btn = gr.Button("temizle_gizli", visible=False, elem_id="temizle-btn")

            gr.HTML("""
            <script>
            function clearChat() {
                var btn = document.getElementById('temizle-btn');
                if (btn) btn.click();
            }
            </script>
            """)

            metrik_goster = gr.HTML(metrik_html_olustur())

            gr.HTML("""
            <div style="text-align:center;color:#334155;font-size:11px;padding:12px 0;">
                © 2024 AI Code Reviewer • NLP Project • YZM358 💜
            </div>
            """)

    analiz_btn.click(analiz_et, [kod_girisi, state], [chatbot, state, metrik_goster])
    sor_btn.click(takip_sorusu, [mesaj_kutusu, state], [chatbot, state, mesaj_kutusu])
    mesaj_kutusu.submit(takip_sorusu, [mesaj_kutusu, state], [chatbot, state, mesaj_kutusu])
    temizle_btn.click(temizle, [state], [chatbot, state])

demo.launch(share=True)

/tmp/ipykernel_44165/656210944.py:198: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title="AI Code Reviewer") as demo:
/tmp/ipykernel_44165/656210944.py:295: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_44165/656210944.py:295: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9770bcac00c4c8f4b9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
